# Assignment 8 

**Student roll number:** 1024170115

## Solution 1

In [ ]:
import re, string, collections
import nltk
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True); nltk.download("stopwords", quiet=True); nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

paragraph = ("I enjoy learning how small robots can help people with everyday tasks. "
"A good robot combines careful sensors with simple, reliable decisions. "
"I especially like projects that use Python because the code is easy to test and improve. "
"When a design fails, the results often show which part needs attention. "
"Building and refining a useful machine is a satisfying way to connect ideas with real life.")
lower_no_punctuation = re.sub(r"[^\w\s]", "", paragraph.lower())
print(lower_no_punctuation)
print("Sentences:", sent_tokenize(paragraph))
print("split():", paragraph.split())
print("word_tokenize():", word_tokenize(paragraph))
stop_words = set(stopwords.words("english"))
tokens = [w for w in word_tokenize(lower_no_punctuation) if w not in stop_words]
freq = collections.Counter(tokens)
print("After stopword removal:", tokens)
print("Frequency:", freq.most_common())
print("split() splits only on whitespace; word_tokenize() separates punctuation and handles common tokenization conventions.")

## Solution 2

In [ ]:
import pandas as pd
import pandas as pd
import pandas as pd
from nltk.stem import PorterStemmer, WordNetLemmatizer
words = re.findall(r"[A-Za-z]+", paragraph.lower())
words = [w for w in words if w not in stop_words]
porter = PorterStemmer(); lemma = WordNetLemmatizer()
stemmed = [porter.stem(w) for w in words]
lemmatized = [lemma.lemmatize(w) for w in words]
comparison = pd.DataFrame({"word": words, "stem": stemmed, "lemma": lemmatized})
display(comparison)
print("Stemming is fast and useful for broad matching, but may produce fragments. Lemmatization uses vocabulary and grammar, so it is usually better when readable word forms matter.")

## Solution 3

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
texts = ["The team built a quiet electric car with a long range.", "The electric bus carries many passengers across the city.", "A software team improved battery monitoring for electric vehicles."]
count_vec = CountVectorizer()
counts = count_vec.fit_transform(texts)
print(pd.DataFrame(counts.toarray(), columns=count_vec.get_feature_names_out(), index=["text 1","text 2","text 3"]))
tfidf_vec = TfidfVectorizer()
tfidf = tfidf_vec.fit_transform(texts)
terms = tfidf_vec.get_feature_names_out()
for i, row in enumerate(tfidf.toarray(), 1):
    top = row.argsort()[-3:][::-1]
    print(f"Text {i} top TF-IDF terms:", [(terms[j], round(row[j], 3)) for j in top])
print("Higher TF-IDF terms distinguish a text from the other texts in this small collection.")

## Solution 4

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
tech_a = "Artificial intelligence learns patterns from examples. It can classify images, recommend content, and support decisions. Good data and careful evaluation help improve an AI system. These tools are used in health, education, transport, and research."
tech_b = "Blockchain stores records in connected blocks shared across a network. It can help verify transactions and track changes. A distributed ledger is useful when several parties need a common history. These systems are used in finance, supply chains, and digital identity."
def clean_tokens(text):
    return set(re.findall(r"[a-z]+", text.lower()))
set_a, set_b = clean_tokens(tech_a), clean_tokens(tech_b)
jaccard = len(set_a & set_b) / len(set_a | set_b)
tfidf_pair = TfidfVectorizer().fit_transform([tech_a, tech_b])
cosine = cosine_similarity(tfidf_pair[0], tfidf_pair[1])[0,0]
print("Jaccard:", jaccard, "Cosine:", cosine)
print("Jaccard treats each distinct word equally. TF-IDF cosine also uses term weighting and can reflect repeated or informative words, so it is more nuanced for these short descriptions; both remain sensitive to vocabulary overlap.")

## Solution 5

In [ ]:
!pip -q install wordcloud textblob

In [ ]:
from textblob import TextBlob
reviews = ["The headphones sound clear and feel comfortable all day.", "The battery life is excellent and setup was easy.", "The plastic feels cheap and one side stopped working.", "The package arrived late and support did not solve my issue.", "The product is fine for the price."]
def classify_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    label = "Positive" if polarity > 0.05 else "Negative" if polarity < -0.05 else "Neutral"
    return polarity, TextBlob(text).sentiment.subjectivity, label
review_df = pd.DataFrame([{"review": r, "polarity": classify_sentiment(r)[0], "subjectivity": classify_sentiment(r)[1], "label": classify_sentiment(r)[2]} for r in reviews])
display(review_df)
positive_text = " ".join(review_df.loc[review_df.label == "Positive", "review"])
try:
    from wordcloud import WordCloud
    import matplotlib.pyplot as plt
    plt.imshow(WordCloud(width=800, height=400, background_color="white").generate(positive_text))
    plt.axis("off"); plt.show()
except ImportError:
    print("Install wordcloud in Colab with: !pip install wordcloud")

## Solution 6

In [ ]:
training_text = ("Robots can help people with repetitive tasks in factories, hospitals, and homes. A useful robot senses its surroundings, follows clear instructions, and responds safely when something changes. "
"Python makes it easier to build small experiments because the language is readable and has many practical libraries. Each experiment can be measured, reviewed, and improved using feedback. "
"Careful testing helps a machine behave reliably, while simple designs are often easier to understand and maintain. A good project starts with a real need, uses thoughtful data, and grows through patient practice and steady improvement.")
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

tok = Tokenizer()
tok.fit_on_texts([training_text])
sequences = []
for line in training_text.split(". "):
    seq = tok.texts_to_sequences([line])[0]
    for end in range(2, len(seq)+1): sequences.append(seq[:end])
max_len = max(map(len, sequences))
seqs = pad_sequences(sequences, maxlen=max_len, padding="pre")
X_train, y_train = seqs[:, :-1], seqs[:, -1]
model = Sequential([Embedding(len(tok.word_index)+1, 24, input_length=max_len-1), LSTM(48), Dense(len(tok.word_index)+1, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, epochs=100, verbose=0)

def generate(seed, count=12):
    text = seed
    for _ in range(count):
        seq = pad_sequences(tok.texts_to_sequences([text]), maxlen=max_len-1, padding="pre")
        probs = model.predict(seq, verbose=0)[0]
        next_id = int(probs.argmax())
        next_word = tok.index_word.get(next_id, "")
        if not next_word: break
        text += " " + next_word
    return text
print(generate("robots can"))
print(generate("python makes"))